# Digit from Image CNN

Alex Keyken 15/11/26

Train a small CNN to classify sudoku cell images (0 = blank, 1-9 = digits).

Pipeline:  generate_digits.py  ->  digits.npz  ->  train_cnn.py  ->  digit_cnn.pt

The script trains on the training fonts and, after every epoch, measures
accuracy on the held-out validation fonts. The epoch with the best validation
accuracy is saved to the checkpoint. At the end it prints per-class accuracy and
the most common mix-ups, and saves training curves and a picture of the
validation images the model got wrong.

Reviewed and optimized using Claude Code.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import torch
from torch import nn
from torch.autograd import grad
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

# setup gpu as device to move models and data onto
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

# setup seed for reproducibility
SEED = 123

torch.manual_seed(SEED)
np.random.seed(SEED)

mps


# Build CNN

In [2]:
def to_tensor(images: np.ndarray) -> torch.Tensor:
    """Convert uint8 images of shape (N, H, W) to a normalised float tensor (N, 1, H, W)."""
    x = torch.from_numpy(images).float().div(255.0)

    # inputs centered on 0 with small range speed up training,
    # so convert data range from [0,1] to [-1,1]
    # unsqueeze(1) since a grayscale img has one channel
    return (2*x - 1).unsqueeze(1)

class DigitCNN(nn.Module):
    """Two conv blocks (conv, batch norm, ReLU, max-pool) and a small classifier."""
    def __init__(self, n_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), # 28 -> 14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

# Train CNN

In [3]:
def run_epoch(model: nn.Module, loader: DataLoader, device: torch.device,
              loss_fn: nn.Module, optimizer: torch.optim.Optimizer | None = None
              ) -> tuple[float, float]:
    """
    One pass over the loader. Trains the model if an optimizer is given, otherwise only
    evaluates the performance. Returns (average loss, accuracy).
    """
    training = optimizer is not None
    model.train(training)
    total_loss, n_correct, n_seen = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = loss_fn(logits, labels)
            
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            total_loss += loss.item() * len(labels) # convert from average to batch total
            n_correct += (logits.argmax(dim=1) == labels).sum().item() # pick the most probable guess then compare to label
            n_seen += len(labels)
    return total_loss/n_seen, n_correct/n_seen

@torch.no_grad()
def predict(model: nn.Module, images: np.ndarray, device: torch.device,
            batch_size: int = 512) -> tuple[np.ndarray, np.ndarray]:
    """Classify uint8 images of shape (N, H, W). Returns (labels, confidences)."""
    model.eval()
    labels, confidences = [], []
    for start in range(0, len(images), batch_size):
        batch = to_tensor(images[start:start + batch_size]).to(device)
        probabilities = torch.softmax(model(batch), dim=1)
        confidence, label = probabilities.max(dim=1)
        labels.append(label.cpu().numpy()) # GPU tensors cannot be converted directly to numpy
        confidences.append(confidence.cpu().numpy())
    return np.concatenate(labels), np.concatenate(confidences) # flatten list of lists

def save_checkpoint(model: nn.Module, path: Path, epoch: int, val_acc: float) -> None:
    """Save a trained model, including the number of epochs and evaluation accuracy."""
    torch.save({"model_state": model.state_dict(), "epoch": epoch, "val_acc": val_acc}, path)

def load_model(path: Path | str, device: torch.device) -> DigitCNN:
    """Load a trained model, ready for prediction."""
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    model = DigitCNN().to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model

# Report on CNN

In [4]:
def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 10) -> np.ndarray:
    """Returns the confusion matrix for the model predictions, such that matrix[true, predicted] = number of images."""
    conf_matrix = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(conf_matrix, (y_true, y_pred), 1)
    return conf_matrix

def print_report(conf_matrix: np.ndarray, n_worst: int = 5) -> None:
    """Print overall and per-class accuracy, as well as the most common mix-ups."""
    print(f"\nOverall validation accuracy: {conf_matrix.diagonal().sum() / conf_matrix.sum():.4f}")
    print("Per-class accuracy:")
    for c in range(len(conf_matrix)):
        name = "blank" if c == 0 else str(c)
        print(f"  {name:>5}: {conf_matrix[c, c] / conf_matrix[c].sum():.4f}  ({conf_matrix[c].sum()} images)")

    mistakes = conf_matrix.copy()
    np.fill_diagonal(mistakes, 0)
    print("Most common mix-ups (true -> predicted):")
    printed = 0 # tally, in case less than n_worst bad predictions are found
    # flatten 2D arr into sorted list, then reverse to find largest n_worst
    for flat in np.argsort(mistakes, axis=None)[::-1][:n_worst]:
        true, pred = divmod(int(flat), mistakes.shape[1]) # convert flattened list index back to row and col indices
        if mistakes[true, pred] == 0:
            break
        print(f"  {true} -> {pred}: {mistakes[true, pred]}")
        printed += 1
    if printed == 0:
        print("  none")

def save_curves(history: dict[str, list[float]], path: Path) -> None:
    """Plot loss and accuracy per epoch for the training and validation sets."""
    epochs = range(1, len(history["train_loss"]) + 1)
    
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(10, 4))
    ax_loss.plot(epochs, history["train_loss"], label="training")
    ax_loss.plot(epochs, history["val_loss"], label="validation")
    ax_loss.set_xlabel("epoch")
    ax_loss.set_ylabel("loss")
    ax_loss.legend()
    
    ax_acc.plot(epochs, history["train_acc"], label="training")
    ax_acc.plot(epochs, history["val_acc"], label="validation")
    ax_acc.set_xlabel("epoch")
    ax_acc.set_ylabel("accuracy")
    ax_acc.legend()
    
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_error_grid(images: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray,
                    confidence: np.ndarray, path: Path, max_images: int = 24, cols: int = 8) -> None:
    """Save the misclassified images in order of most to least confident mistakes."""
    wrong = np.flatnonzero(y_true != y_pred)
    if len(wrong) == 0:
        print("No misclassified validation images, so no error picture was saved.")
        return None
    wrong = wrong[np.argsort(-confidence[wrong])][:max_images]

    rows = int(np.ceil(len(wrong) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.3, rows * 1.6), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(images[i], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        ax.set_title(f"{y_true[i]} -> {y_pred[i]}", fontsize=9)
    fig.suptitle("Misclassified validation images (true -> predicted), most to least confident")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

# Main

In [12]:
data_path = f"datasets/{SEED}_sample.npz"

data = np.load(data_path)

x_train, y_train = data["x_train"], data["y_train"]
x_val, y_val = data["x_val"], data["y_val"]
print(f"Training images: {len(y_train)}, validation images: {len(y_val)}")

Training images: 25500, validation images: 6400


In [13]:
lr = 0.001
n_epochs = 10

# convert arrs of images and labels to pytorch tensors
train_loader = DataLoader(TensorDataset(to_tensor(x_train), torch.from_numpy(y_train)),
                            batch_size=512, shuffle=True)
val_loader = DataLoader(TensorDataset(to_tensor(x_val), torch.from_numpy(y_val)),
                            batch_size=512)

# init the model, loss function, optimizer and scheduler
model = DigitCNN().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# use scheduler to reduce lr in later epochs when values are close to optimum
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

Model parameters: 421,834


In [14]:
model_path = f"models/{SEED}_{n_epochs}_digitCNN.pt"

history: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, device, loss_fn, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, device, loss_fn)
    scheduler.step()

    for key, value in zip(history, (train_loss, train_acc, val_loss, val_acc)):
        history[key].append(value)
    
    print(f"epoch {epoch:2d}/{n_epochs}  "
          f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, model_path, epoch, val_acc)

print(f"\nBest validation accuracy {best_val_acc:.4f}; model saved to {model_path}")

epoch  1/10  train loss 1.0880 acc 0.6492 | val loss 0.3492 acc 0.9222
epoch  2/10  train loss 0.2578 acc 0.9361 | val loss 0.1790 acc 0.9533
epoch  3/10  train loss 0.1565 acc 0.9608 | val loss 0.1192 acc 0.9664
epoch  4/10  train loss 0.1161 acc 0.9700 | val loss 0.0904 acc 0.9767
epoch  5/10  train loss 0.0931 acc 0.9759 | val loss 0.0847 acc 0.9753
epoch  6/10  train loss 0.0801 acc 0.9789 | val loss 0.0745 acc 0.9775
epoch  7/10  train loss 0.0677 acc 0.9831 | val loss 0.0681 acc 0.9802
epoch  8/10  train loss 0.0605 acc 0.9856 | val loss 0.0651 acc 0.9812
epoch  9/10  train loss 0.0569 acc 0.9862 | val loss 0.0628 acc 0.9817
epoch 10/10  train loss 0.0550 acc 0.9870 | val loss 0.0612 acc 0.9825

Best validation accuracy 0.9825; model saved to models/123_10_digitCNN.pt


In [15]:
report_dir = Path("reports")

model = load_model(model_path, device)
labels, confidence = predict(model, x_val, device)
print_report(confusion_matrix(y_val, labels))

report_dir.mkdir(parents=True, exist_ok=True)
save_curves(history, report_dir / f"{SEED}_training_curves.png")
save_error_grid(x_val, y_val, labels, confidence, report_dir / f"{SEED}_validation_errors.png")
print(f"Saved training curves and error picture to {report_dir.resolve()}")


Overall validation accuracy: 0.9825
Per-class accuracy:
  blank: 1.0000  (640 images)
      1: 0.9516  (640 images)
      2: 0.9906  (640 images)
      3: 0.9781  (640 images)
      4: 0.9875  (640 images)
      5: 0.9938  (640 images)
      6: 0.9953  (640 images)
      7: 0.9656  (640 images)
      8: 0.9734  (640 images)
      9: 0.9891  (640 images)
Most common mix-ups (true -> predicted):
  1 -> 4: 17
  7 -> 1: 15
  3 -> 8: 9
  1 -> 7: 8
  4 -> 1: 5
Saved training curves and error picture to /Users/alexkeyken/projects/sudoku-solver/reports
